# Ensemble Methods - Starter Notebook
This notebook compares different ensemble strategies: Bagging, Boosting (AdaBoost, GradientBoosting), and Voting.

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import (
    AdaBoostClassifier,
    BaggingClassifier,
    GradientBoostingClassifier,
    VotingClassifier
)
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

X_train.shape

In [ ]:
base_tree = DecisionTreeClassifier(max_depth=5, random_state=42)

models = {
    "Bagging": BaggingClassifier(base_tree, n_estimators=50, random_state=42),
    "AdaBoost": AdaBoostClassifier(base_tree, n_estimators=50, random_state=42),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=50, random_state=42),
    "Voting": VotingClassifier(
        estimators=[
            ("bagging", BaggingClassifier(n_estimators=50, random_state=42)),
            ("adaboost", AdaBoostClassifier(n_estimators=50, random_state=42)),
            ("gb", GradientBoostingClassifier(n_estimators=50, random_state=42))
        ]
    )
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    results.append({"Model": name, "Train Acc": train_acc, "Test Acc": test_acc})

results_df = pd.DataFrame(results)
results_df

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
x = range(len(results_df))
plt.bar([i - 0.2 for i in x], results_df["Train Acc"], width=0.4, label="Train Acc")
plt.bar([i + 0.2 for i in x], results_df["Test Acc"], width=0.4, label="Test Acc")
plt.xticks(x, results_df["Model"], rotation=45)
plt.title("Ensemble Methods Comparison")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()